# Procesamiento de lenguaje natural
## LSTM Bot QA

### 1 - Datos
El objecto es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [2]:
!pip install --upgrade --no-cache-dir gdown --quiet

In [61]:
import re

import numpy as np
import pandas as pd
import json

import tensorflow as tf
from tensorflow.keras.utils import pad_sequences
from keras.models import Sequential
from keras.layers import Activation, Dropout, Dense
from keras.layers import Flatten, LSTM, SimpleRNN
from keras.models import Model
from tensorflow.keras.layers import Embedding
from sklearn.model_selection import train_test_split
from keras.layers import Input
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.src.utils import to_categorical


In [11]:
with open("datasets/data_volunteers.json") as f:
    data = json.load(f)
    print(f"dataset cargado, campos: {data[0].keys()}")

dataset cargado, campos: dict_keys(['dialog', 'start_time', 'end_time', 'bot_profile', 'user_profile', 'eval_score', 'profile_match', 'participant1_id', 'participant2_id'])


Limpiaremos el dataset de caracteres y abreviaturas. Además acondicionaremos la entrada al encoder y al decoder así como también la salida de este último. Para la entrada del decoder agregaremos un caracter que indique el comienzo de un mensaje (<sos>), para la salida agregaremos uno que indique el fín (<eos>). De esta forma el modelo aprenderá a cortar oraciones.

In [31]:
chat_in = []
chat_out = []

input_sentences = []
output_sentences = []
output_sentences_inputs = []
max_len = 10

def clean_text(txt):
    txt = txt.lower()
    txt.replace("\'d", " had")
    txt.replace("\'s", " is")
    txt.replace("\'m", " am")
    txt.replace("don't", "do not")
    txt = re.sub(r'\W+', ' ', txt)

    return txt

for line in data:
    for i in range(len(line['dialog'])-1):

        chat_in = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i+1]['text'])

        if len(chat_in.split()) >= max_len or len(chat_out.split()) >= max_len:
            continue

        input_sentence, output = chat_in, chat_out

        # output sentence (decoder_output) tiene <eos>
        output_sentence = output + ' <eos>'
        # output sentence input (decoder_input) tiene <sos>
        output_sentence_input = '<sos> ' + output

        input_sentences.append(input_sentence)
        output_sentences.append(output_sentence)
        output_sentences_inputs.append(output_sentence_input)

print(f"Cantidad de rows utilizadas: {len(input_sentences)} \n")
print(f"encoder_input: {input_sentences[1]}")
print(f"decoder_training_input: {output_sentences_inputs[1]}")
print(f"decoder_training_output: {output_sentences[1]}")


Cantidad de rows utilizadas: 9092 

encoder_input: hi how are you 
decoder_training_input: <sos> not bad and you 
decoder_training_output: not bad and you  <eos>


### 2 - Preprocesamiento
Realizar el preprocesamiento necesario para obtener:
- word2idx_inputs, max_input_len
- word2idx_outputs, max_out_len, num_words_output
- encoder_input_sequences, decoder_output_sequences, decoder_targets

In [32]:
# Definir el tamaño máximo del vocabulario
MAX_VOCAB_SIZE = 8000

In [58]:
print("INPUT")

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)

word2idx_inputs = input_tokenizer.word_index
print("Palabras en el vocabulario:", len(word2idx_inputs))

max_input_len = max(len(sen) for sen in input_integer_seq)
print("Sentencia de entrada más larga:", max_input_len)

print("\nOUTPUT")

output_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='')
output_tokenizer.fit_on_texts(output_sentences + output_sentences_inputs)
output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)

word2idx_outputs = output_tokenizer.word_index
print("Palabras en el vocabulario:", len(word2idx_outputs))

num_words_output = len(word2idx_outputs) + 1
max_out_len = max(len(sen) for sen in output_integer_seq)
print("Sentencia de entrada más larga:", max_out_len)

INPUT
Palabras en el vocabulario: 2724
Sentencia de entrada más larga: 9

OUTPUT
Palabras en el vocabulario: 2732
Sentencia de entrada más larga: 10


In [59]:
print("ENCODER INPUT")
encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
print(f"encoder_input_sequences.shape: {encoder_input_sequences.shape}")
print(f"message: {input_sentences[1]}")
print(f"luego de hacer tokenizarla y hacerle padding: {encoder_input_sequences[1]}")

print("\nDECODER INPUT")
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
print(f"decoder_input_sequences.shape: {decoder_input_sequences.shape}")
print(f"message: {output_sentences_inputs[1]}")
print(f"luego de hacer tokenizarla y hacerle padding: {decoder_input_sequences[1]}")

print("\nDECODER OUTPUT")
decoder_output_sequences = pad_sequences(output_output_integer_seq, maxlen=max_out_len, padding='post')
print(f"decoder_output_sequences.shape: {decoder_output_sequences.shape}")
print(f"message: {output_sentences[1]}")
print(f"luego de hacer tokenizarla y hacerle padding: {decoder_output_sequences[1]}")

ENCODER INPUT
encoder_input_sequences.shape: (9092, 9)
message: hi how are you 
luego de hacer tokenizarla y hacerle padding: [ 0  0  0  0  0 16 11  7  2]

DECODER INPUT
decoder_input_sequences.shape: (9092, 10)
message: <sos> not bad and you 
luego de hacer tokenizarla y hacerle padding: [  2  26 240  32   4   0   0   0   0   0]

DECODER OUTPUT
decoder_output_sequences.shape: (9092, 10)
message: not bad and you  <eos>
luego de hacer tokenizarla y hacerle padding: [ 26 240  32   4   1   0   0   0   0   0]


Crearemos una nueva variable decoder_tagets con la representación de decoder_output_sequences pasadas por OHE. Esto es necesario ya que luego usaremos una capa densa con una función softmax de activación. De esta forma tendremos una matrix de 3 dimensiones:

La primer dimensión identifica la cantidad de documentos en el corpus
La segunda es la cantidad de palabras que pueden aparecer en un mensaje de salida del chatbot
La tercera es la cantidad de palabras en el vocabulario de los mensajes de salida. Esta dimensión será un vector de 1 y 0s dónde el 1 estará en la columna que identifica la palabra 

In [62]:
decoder_targets = to_categorical(decoder_output_sequences, num_classes=num_words_output)
print(f'decoder_targets.shape: {decoder_targets.shape}')

decoder_targets.shape: (9092, 10, 2733)


### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores